# X5_Q2 — Core descriptivo (Pearson, Spearman, coseno, regímenes, estabilidad, export)

Segundo notebook de **X5_alt_v2** (ver
[`docs/plans/X5_alternativo.md`](../docs/plans/X5_alternativo.md) sección 10,
documento rector [`docs/plans/x5_contexto_conversacion.md`](../docs/plans/x5_contexto_conversacion.md)).
TODOs en `docs/tracking/todos.md`, sección X5_alt_v2.

Consume la tabla de estado que construye `X5_Q1.ipynb`
(`resources/x5_alt/{valor}_tabla_estado_v2.csv`: precio + X3 + X2 + variables
de drawdown/volatilidad/liquidez/fundamentales/ruido/tendencia) y agrega
sobre ella el análisis descriptivo variable↔precio — correlación clásica
(Pearson/Spearman), coseno `cos(Pv,Xv)`, no linealidades, ceteris paribus,
redundancia, regímenes y estabilidad — más el export estructurado de
métricas que retoma `X5_Q3.ipynb`.

Mismo input por lista `valores` que `X5_Q1.ipynb` (no el `valor` singular de v1).

In [1]:
valores = ['BTCUSD']  # lista de activos; default solo BTCUSD (ver plan X5_alternativo.md §10.2)

## 0. Preparación

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# cwd = scripts/ al correr en Jupyter; fallback a __file__ por si se ejecuta como script
SCRIPTS_DIR = Path.cwd() if (Path.cwd() / 'config.py').exists() else Path(__file__).resolve().parent
sys.path.insert(0, str(SCRIPTS_DIR))
import config as cfg  # noqa: E402

pd.set_option('display.width', 140)  # evita que pandas trunque columnas al imprimir DataFrames anchos

print(f"Activos: {valores}")

Activos: ['BTCUSD']


## 1. Carga de la tabla de estado de X5_Q1 (Q2-1)

Lee `{valor}_tabla_estado_v2.csv` por cada activo de `valores` — generada
por `X5_Q1.ipynb` (Sección 10), no recalculada acá. Si no existe, correr
`X5_Q1.ipynb` primero.

In [3]:
def _cargar_tabla_estado(valor):
    path = cfg.CARPETA_X5_ALT / f'{valor}_tabla_estado_v2.csv'
    if not path.exists():
        raise FileNotFoundError(f"No existe {path} — correr X5_Q1.ipynb primero para generarla.")
    return pd.read_csv(path, parse_dates=['DateTime'])


tablas = {valor: _cargar_tabla_estado(valor) for valor in valores}

for valor, tabla in tablas.items():
    print(f"{valor}: {tabla.shape[0]} filas x {tabla.shape[1]} columnas — {tabla['DateTime'].min()} → {tabla['DateTime'].max()}")

BTCUSD: 40283 filas x 72 columnas — 2021-10-27 16:00:00 → 2026-06-02 21:00:00


## 2. Gráficos a doble eje precio + variable (Q2-2)

**Objetivo**: inspeccionar visualmente el co-movimiento entre el precio (`Close`) y
cada variable de estado de la tabla, conservando la escala nativa de cada una en
vez de forzarlas a una sola unidad.

**Pregunta**: ¿la variable se mueve en sincronía, con rezago o de forma
independiente respecto al precio, vista en su propia escala?

**Metodología**: por cada variable de la tabla (todas las columnas salvo las de
precio/volumen crudo — `DateTime`, `Open`, `High`, `Low`, `Close`, `Tick_Volume`,
`Spread`, `Real_Volume`), un gráfico de doble eje: precio en el eje izquierdo,
variable en el eje derecho, con **ambos ejes centrados en su propia media** (mismo
punto medio visual) para que una desviación de la variable respecto a su media sea
comparable a simple vista con una desviación del precio respecto a la suya, sin
que la diferencia de unidades/magnitud lo distorsione. Para las variables
categóricas (`regimen_tendencia`, `ruptura_rango`, `fundamental_direccion_cambio`)
el eje doble no es interpretable como línea continua en su escala nativa — ahí se
usa el caso al que se refiere el TO DO como "series normalizadas superpuestas
cuando las escalas sean incompatibles": precio y variable, ambas como z-score, en
un solo eje.

**Interpretación**: exploración visual únicamente — adelanta intuición sobre qué
variables acompañan o se desacoplan del precio, pero la cuantificación
(Pearson/Spearman/coseno) es responsabilidad de Q2-3/Q2-4, no de este bloque.

**Limitaciones**: con variables 100% nulas en el snapshot actual de Mac (`x2_score`
y sus derivadas para BTCUSD, ver catálogo de Q1 — sin solapamiento temporal con
datos reales de X2) no hay superposición no nula con el precio — el gráfico se
omite y se imprime un aviso en vez de generar un panel vacío.

In [4]:
import matplotlib.pyplot as plt

EXCLUIR_PRECIO = {'DateTime', 'Open', 'High', 'Low', 'Close', 'Tick_Volume', 'Spread', 'Real_Volume'}
CATEGORICAS = {'regimen_tendencia', 'ruptura_rango', 'fundamental_direccion_cambio'}
PRECIO_COL = 'Close'


def _ylim_centrado(serie, k=1.05):
    """Límites del eje centrados en la media de `serie`, con margen simétrico según su rango real."""
    media = serie.mean()
    margen = max(serie.max() - media, media - serie.min()) * k
    if not np.isfinite(margen) or margen == 0:
        margen = 1.0
    return media - margen, media + margen


def _serie_numerica(serie):
    """Codifica categóricas (ej. regimen_tendencia) a enteros para poder graficarlas/normalizarlas."""
    if pd.api.types.is_numeric_dtype(serie):
        return serie
    codigos, _ = pd.factorize(serie)
    return pd.Series(codigos, index=serie.index).replace(-1, np.nan)


def _zscore(serie):
    std = serie.std()
    if not std or not np.isfinite(std):
        return serie * 0
    return (serie - serie.mean()) / std


def plot_doble_eje(df, variable, precio_col=PRECIO_COL):
    """Q2-2: precio (eje izq.) vs. variable (eje der.), ambos centrados en su propia media.
    Si `variable` es categórica, usa un solo eje con ambas series normalizadas (z-score)."""
    serie_variable = _serie_numerica(df[variable])
    sub = pd.DataFrame({'DateTime': df['DateTime'], precio_col: df[precio_col], variable: serie_variable}).dropna()
    if sub.empty:
        print(f"{variable}: sin superposición temporal no nula con {precio_col} — se omite.")
        return False

    fig, ax1 = plt.subplots(figsize=(14, 4))

    if variable in CATEGORICAS:
        ax1.plot(sub['DateTime'], _zscore(sub[precio_col]), color='#1f77b4', linewidth=0.8, label=precio_col)
        ax1.plot(sub['DateTime'], _zscore(sub[variable]), color='#d62728', linewidth=0.8, alpha=0.8, label=variable)
        ax1.set_ylabel('z-score')
        ax1.set_title(f"{precio_col} vs. {variable} \u2014 normalizadas superpuestas (categ\u00f3rica, eje doble no aplica)")
        ax1.legend(fontsize=8)
    else:
        ax1.plot(sub['DateTime'], sub[precio_col], color='#1f77b4', linewidth=0.8)
        ax1.set_ylabel(precio_col, color='#1f77b4')
        ax1.tick_params(axis='y', labelcolor='#1f77b4')
        ax1.set_ylim(_ylim_centrado(sub[precio_col]))

        ax2 = ax1.twinx()
        ax2.plot(sub['DateTime'], sub[variable], color='#d62728', linewidth=0.8, alpha=0.85)
        ax2.set_ylabel(variable, color='#d62728')
        ax2.tick_params(axis='y', labelcolor='#d62728')
        ax2.set_ylim(_ylim_centrado(sub[variable]))

        ax1.set_title(f"{precio_col} (izq.) vs. {variable} (der.) \u2014 ejes centrados en su propia media")

    ax1.set_xlabel('DateTime')
    plt.tight_layout()
    plt.show()
    return True

In [5]:
for valor, tabla in tablas.items():
    variables_q2_2 = [c for c in tabla.columns if c not in EXCLUIR_PRECIO]
    graficados = sum(plot_doble_eje(tabla, variable) for variable in variables_q2_2)
    print(f"{valor}: {graficados}/{len(variables_q2_2)} variables graficadas (resto omitido por falta de datos no nulos).")

x2_score: sin superposición temporal no nula con Close — se omite.
x2_score_cross: sin superposición temporal no nula con Close — se omite.
x2_score_tendencia: sin superposición temporal no nula con Close — se omite.


fundamental_cambio: sin superposición temporal no nula con Close — se omite.
fundamental_direccion_cambio: sin superposición temporal no nula con Close — se omite.
fundamental_magnitud_cambio: sin superposición temporal no nula con Close — se omite.


fundamental_estabilidad_90d: sin superposición temporal no nula con Close — se omite.
fundamental_contradiccion: sin superposición temporal no nula con Close — se omite.


BTCUSD: 56/64 variables graficadas (resto omitido por falta de datos no nulos).
